In [2]:
import numpy as np
import pandas as pd
import geopandas as gpd
import plotly.express as px
import plotly.graph_objects as go
import streamlit as st
from pathlib import Path

# ==================== KONFIGURASI ====================
ROOT = Path('/content/drive/MyDrive/Logika Ui')
AKAR = ROOT / 'KTI_KARHUTLA'
HASIL = AKAR / 'hasil'
LAIN = AKAR / 'lainnya'

GADM_SHP = ROOT / 'GADM 2' / 'gadm41_IDN_2.shp'
GAMBUT_GEOJSON = LAIN / 'Indonesia_Peat_Lands.geojson'
TABEL_UTAMA = HASIL / 'tabel_gabungan_nasional_semua_bulan.csv'
PROB_FINAL = HASIL / 'prob_final_nasional.npz'
GAMBUT_KAB = HASIL / 'gambut_kabkota.csv'
PRIORITAS_HARIAN = HASIL / 'prioritas_harian.csv'
SKOR_PROV = HASIL / 'skill_per_provinsi.csv'
SKOR_BLN = HASIL / 'skill_per_bulan.csv'
BACKTEST = HASIL / 'backtest_skor_harian.csv'

TAU = 0.05
PRIORITAS = ['Riau', 'Jambi', 'Sumatera Selatan',
             'Kalimantan Barat', 'Kalimantan Tengah', 'Kalimantan Selatan']

st.set_page_config(page_title='fuwa', page_icon='🔥', layout='wide')

# ==================== MUAT DATA ====================
@st.cache_data
def muat_peta():
    g = gpd.read_file(GADM_SHP)[['GID_2', 'NAME_1', 'NAME_2', 'geometry']]
    g['geometry'] = g.geometry.make_valid()
    return g

@st.cache_data
def muat_prioritas():
    p = pd.read_csv(PRIORITAS_HARIAN, parse_dates=['tanggal'])
    return p

@st.cache_data
def muat_harian_lengkap():
    # titik_panas asli per hari untuk grafik detail
    t = pd.read_csv(TABEL_UTAMA, parse_dates=['tanggal'],
                    usecols=['tanggal', 'GID_2', 'titik_panas'])
    return t

@st.cache_data
def muat_gambut():
    if GAMBUT_KAB.exists():
        return pd.read_csv(GAMBUT_KAB)
    return None

@st.cache_data
def muat_kalibrasi():
    z = np.load(PROB_FINAL)
    t = pd.read_csv(TABEL_UTAMA, parse_dates=['tanggal'],
                    usecols=['hujan_rata', 'frac_hujan'])
    kelas = np.where((t['hujan_rata'] * 6) >= 1, 2,
                     np.where(t['frac_hujan'] >= TAU, 1, 0))
    p1 = z['PD_kalibrasi'][:, 1]
    d = pd.DataFrame({'p': p1, 'y': (kelas == 1).astype(float)})
    d['bin'] = pd.qcut(d['p'], 10, duplicates='drop')
    rel = d.groupby('bin', observed=True)[['p', 'y']].mean().reset_index(drop=True)
    return rel

@st.cache_data
def muat_backtest():
    if not BACKTEST.exists():
        return None
    b = pd.read_csv(BACKTEST, index_col=0, parse_dates=True, header=[0, 1])
    return b

g = muat_peta()
p = muat_prioritas()
tpanas = muat_harian_lengkap()
gambut = muat_gambut()
rel = muat_kalibrasi()
bt = muat_backtest()

tanggal_valid = sorted(p.loc[p['api_7h'].notna(), 'tanggal'].dt.date.unique())

# ==================== SIDEBAR ====================
st.sidebar.title('🔥 fuwa')
tgl = st.sidebar.selectbox('Tanggal', tanggal_valid, index=len(tanggal_valid) - 1)
pakai_gambut = st.sidebar.toggle('Sertakan bobot gambut', value=True)
cakupan = st.sidebar.radio('Cakupan', ['Nasional', 'Enam provinsi prioritas'])
skor_kolom = 'skor' if pakai_gambut else 'skor_tanpa_gambut'

st.sidebar.divider()
kab_pilihan = st.sidebar.selectbox(
    'Detail kabupaten/kota',
    sorted(p[['GID_2', 'NAME_2']].drop_duplicates().apply(
        lambda r: f"{r['NAME_2']} ({r['GID_2']})", axis=1).tolist())
    if 'NAME_2' in p.columns else sorted(p['GID_2'].unique()))
gid_pilihan = kab_pilihan.split('(')[-1].rstrip(')') if '(' in kab_pilihan else kab_pilihan

st.title('🔥 fuwa - dashboard skor prioritas kebakaran')

tab1, tab2, tab3, tab4 = st.tabs(
    ['🗺️ Peta peringkat', '📋 10 teratas', '📈 Detail wilayah', '✅ Seberapa bisa dipercaya'])

# ==================== TAB 1: PETA ====================
with tab1:
    h = p[p['tanggal'].dt.date == tgl].copy()
    if cakupan == 'Enam provinsi prioritas':
        h = h[h['NAME_1'].isin(PRIORITAS)]
    h = h.merge(g[['GID_2', 'NAME_2']], on='GID_2', how='left')
    geo = g.merge(h[['GID_2', skor_kolom, 'NAME_1', 'NAME_2']],
                  on='GID_2', how='inner')

    fig = px.choropleth_mapbox(
        geo, geojson=geo.__geo_interface__, locations='GID_2', featureidkey='properties.GID_2',
        color=skor_kolom, color_continuous_scale='YlOrRd',
        hover_name='NAME_2',
        hover_data={'NAME_1': True, skor_kolom: ':.3f', 'GID_2': False},
        mapbox_style='carto-positron', zoom=3.2,
        center={'lat': -2.5, 'lon': 118},
        labels={skor_kolom: 'skor'})
    fig.update_layout(margin=dict(l=0, r=0, t=0, b=0), height=600)
    st.plotly_chart(fig, use_container_width=True)

    c1, c2, c3 = st.columns(3)
    c1.metric('Wilayah terdata hari ini', int(h['GID_2'].nunique()))
    c2.metric('Skor median', round(float(h[skor_kolom].median()), 3))
    c3.metric('Skor maksimum', round(float(h[skor_kolom].max()), 3))

# ==================== TAB 2: TABEL ====================
with tab2:
    kolom_tampil = ['GID_2', 'NAME_1', 'NAME_2', 'p_K1', 'api_7h', 'persen_gambut', skor_kolom]
    top10 = h.nlargest(10, skor_kolom)[kolom_tampil].reset_index(drop=True)
    top10.index += 1
    top10 = top10.rename(columns={
        'NAME_1': 'Provinsi', 'NAME_2': 'Kab/Kota', 'p_K1': 'Peluang K1',
        'api_7h': 'Titik panas 7 hari', 'persen_gambut': '% gambut',
        skor_kolom: 'Skor'})
    top10['Peluang K1'] = (top10['Peluang K1'] * 100).round(1).astype(str) + ' %'
    top10['% gambut'] = top10['% gambut'].round(2)
    top10['Skor'] = top10['Skor'].round(3)
    st.subheader(f'10 teratas - {tgl}' + (' (dengan bobot gambut)' if pakai_gambut else ' (tanpa gambut)'))
    st.dataframe(top10.drop(columns='GID_2'), use_container_width=True)
    st.download_button('Unduh CSV hari ini',
                       h[kolom_tampil].to_csv(index=False).encode(),
                       file_name=f'fuwa_{tgl}.csv')

# ==================== TAB 3: DETAIL ====================
with tab3:
    d = p[p['GID_2'] == gid_pilihan].sort_values('tanggal').tail(90)
    t = tpanas[tpanas['GID_2'] == gid_pilihan].sort_values('tanggal').tail(90)
    nama_kab = g.loc[g['GID_2'] == gid_pilihan, 'NAME_2']
    nama_kab = nama_kab.iloc[0] if len(nama_kab) else gid_pilihan
    st.subheader(f'{nama_kab} - 90 hari terakhir')

    f1 = go.Figure()
    f1.add_trace(go.Scatter(x=t['tanggal'], y=t['titik_panas'],
                            name='Titik panas', line=dict(color='firebrick')))
    f1.update_layout(title='Titik panas harian', height=260,
                     margin=dict(l=0, r=0, t=40, b=0))
    st.plotly_chart(f1, use_container_width=True)

    f2 = go.Figure()
    f2.add_trace(go.Scatter(x=d['tanggal'], y=d['p_K1'],
                            name='p_K1', line=dict(color='steelblue')))
    f2.update_layout(title='Peluang K1 (awan konvektif lokal, model C)',
                     yaxis=dict(tickformat='.0%'), height=260,
                     margin=dict(l=0, r=0, t=40, b=0))
    st.plotly_chart(f2, use_container_width=True)

    f3 = go.Figure()
    f3.add_trace(go.Scatter(x=d['tanggal'], y=d['skor'],
                            name='Skor (dengan gambut)', line=dict(color='darkorange')))
    f3.add_trace(go.Scatter(x=d['tanggal'], y=d['skor_tanpa_gambut'],
                            name='Skor (tanpa gambut)', line=dict(color='gray', dash='dot')))
    f3.update_layout(title='Skor prioritas', height=260,
                     margin=dict(l=0, r=0, t=40, b=0))
    st.plotly_chart(f3, use_container_width=True)

# ==================== TAB 4: KREDIBILITAS ====================
with tab4:
    st.subheader('Skill model per provinsi (skill 3 kelas vs klimatologi)')
    if SKOR_PROV.exists():
        sp = pd.read_csv(SKOR_PROV, index_col=0)
        fprov = px.bar(sp.sort_values('skill', ascending=True).tail(34),
                       x='skill', y=sp.sort_values('skill', ascending=True).tail(34).index,
                       orientation='h', height=800,
                       labels={'skill': 'Skill', 'index': 'Provinsi'})
        st.plotly_chart(fprov, use_container_width=True)
        st.caption('Skill 0 berarti sama dengan klimatologi, makin besar makin baik.')
    else:
        st.info('File skill_per_provinsi.csv belum ada, jalankan fuwa_model (1).ipynb.')

    st.subheader('Skill per bulan')
    if SKOR_BLN.exists():
        sb = pd.read_csv(SKOR_BLN, index_col=0)
        fbln = px.bar(sb.reset_index().rename(columns={'index': 'bulan'}),
                      x='bulan', y='skill',
                      labels={'bulan': 'Bulan', 'skill': 'Skill'}, height=300)
        st.plotly_chart(fbln, use_container_width=True)

    st.subheader('Grafik kalibrasi peluang K1')
    fcal = go.Figure()
    fcal.add_trace(go.Scatter(x=[0, 0.5], y=[0, 0.5], mode='lines',
                              name='Sempurna', line=dict(dash='dash', color='gray')))
    fcal.add_trace(go.Scatter(x=rel['p'], y=rel['y'], mode='lines+markers',
                              name='Model D terkalibrasi'))
    fcal.update_layout(xaxis_title='Peluang K1 prediksi',
                       yaxis_title='Proporsi K1 teramati', height=400)
    st.plotly_chart(fcal, use_container_width=True)

    st.subheader('Hasil backtest (Jun-Okt, rata-rata harian)')
    if bt is not None:
        ringkas = bt.xs('U', level=1, axis=1).mean().sort_values(ascending=False)
        ringkas = ringkas.rename('U = api 7 hari depan x proporsi K1').round(3)
        st.table(ringkas)
        st.caption('U mengukur seberapa sering daftar prioritas menangkap '
                   'daerah yang benar-benar kebakaran (log api depan x K1). '
                   'Acak = 0.177, skor tanpa gambut = 0.985, skor dengan gambut = 0.866.')
    else:
        st.info('File backtest_skor_harian.csv belum ada, jalankan fuwa_cek_nrt_dan_backtest.ipynb.')

    st.subheader('Catatan data')
    st.markdown(
        '- **Peta gambut** memakai peta gambut 2019 (Indonesia_Peat_Lands.geojson). '
        'Peta gambut yang lebih baru akan menggeser bobot r_gambut, tapi uji sensitivitas '
        'menunjukkan peringkat top-10 hampir tidak berubah (irisan rata-rata 0.957 '
        'bahkan pada skenario ekstrem kehilangan gambut).\n'
        '- **Data FIRMS NRT** (pengganti SP) baru tersedia mulai **1 Juli 2026**. '
        'Sebelum itu pakai SP. Titik di sel statis (permukiman, industri) disaring.\n'
        '- **FIRMS kosong 54 hari** di seluruh periode, di antaranya akhir Januari 2026 '
        '(kelengkapan Himawari juga turun di awal 2023).\n'
        '- **Himawari kosong sekitar 10%** dari baris, jadi model D sering jatuh ke '
        'model C sebagai cadangan.')

2026-10-25 12:05:32.414 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-25 12:05:32.416 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-10-25 12:05:32.417 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-10-25 12:05:32.419 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-10-25 12:05:32.421 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-10-25 12:05:32.422 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-10-25 12:05:32.425 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-10-25 12:05:32.427 WARNING streamlit.runtime.caching.cache_d

DataSourceError: /content/drive/MyDrive/Logika Ui/GADM 2/gadm41_IDN_2.shp: No such file or directory

In [1]:
!pip -q install streamlit pyngrok geopandas plotly pyogrio
# simpan kode di atas sebagai fuwa_dashboard.py lewat %%writefile, lalu:
!streamlit run fuwa_dashboard.py & npx localtunnel --port 8501

Usage: streamlit run [OPTIONS] [TARGET] [ARGS]...
Try 'streamlit run --help' for help.

Error: Invalid value: File does not exist: fuwa_dashboard.py
⠙⠹⠸⠼⠴Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) y

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏your url is: https://loud-carrots-say.loca.lt
^C
